
# Planet PSScene 2020 — AOI Search → Batched Orders (clip → reproject → COG) → Download

This notebook automates:
1) Reading your AOI (GeoJSON).  
2) Searching **PSScene (PlanetScope)** scenes in **2020** with a cloud filter.  
3) Placing **batched Orders** with a **toolchain**: `clip → reproject → COG`.  
4) Downloading into a tidy local folder structure you can feed into your labeling / mask-rasterization pipeline.

> **Auth:** Run `planet auth login` once so the SDK can pick it up.  
> **Quota:** Running orders consumes quota; double‑check filters before executing.


In [3]:
# %pip install -q planet geopandas shapely tqdm pandas

import os
import json
from pathlib import Path
from datetime import datetime, date

from tqdm import tqdm
import pandas as pd

from planet import Planet, data_filter, order_request, reporting

try:
    import geopandas as gpd  # optional
except Exception:
    gpd = None

AOI_DIR = Path("D:\\Thesis\\glacial_lake_detection_thesis\\Training\\1 Getting geojson for imageries\\aoi_geojson_convexhull")   # set this to your folder path
GEOJSON_PATTERN = "*.geojson"
OUT_DIR = Path("./planet_downloads_PSScene_rgb_2020")
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2020
DATE_START = f"{YEAR}-09-01"
DATE_END = f"{YEAR}-09-30"
MAX_CLOUD = 0.2

PRIMARY_BUNDLE = "visual"
#PRIMARY_BUNDLE = "analytic_8b_sr_udm2"
FALLBACK_BUNDLES = ["analytic_sr_udm2", "analytic_8b_udm2"]

REPROJECT_EPSG = "EPSG:4326"
MAX_ITEMS_PER_ORDER = 400
WAIT_MAX_ATTEMPTS = 600
ZIP_EACH_ORDER = False


In [4]:
!planet auth login
pl = Planet()
print("Planet client ready. If this fails, run `!planet auth login`.")


Logging in with authentication profile planet-user...
Opening browser to login.
Confirm the authorization code when prompted: WPWT-BDSS

Login succeeded.
Planet client ready. If this fails, run `!planet auth login`.


In [5]:
def read_aoi_geojson(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"AOI GeoJSON not found: {path}")
    obj = json.loads(path.read_text())
    if obj.get("type") == "FeatureCollection":
        geom = obj["features"][0]["geometry"]
    elif obj.get("type") == "Feature":
        geom = obj["geometry"]
    else:
        geom = obj
    if geom.get("type") not in {"Polygon", "MultiPolygon"}:
        raise ValueError("AOI must be a Polygon or MultiPolygon geometry.")
    return geom


In [6]:
def process_one_aoi(aoi_path: str|Path):
    """Run the same single-AOI flow for a given GeoJSON path.
    Creates a subfolder under OUT_DIR using the AOI filename stem.
    Returns list of created order IDs.
    """
    
    aoi_path = Path(aoi_path)
    aoi_geom = read_aoi_geojson(aoi_path)

    # Ensure sub-output dir per AOI
    aoi_name = aoi_path.stem
    aoi_out = OUT_DIR / aoi_name
    aoi_out.mkdir(parents=True, exist_ok=True)

    # ---- Search (mirrors earlier search cell) ----
    items = []
    sr_or = data_filter.or_filter([
    data_filter.asset_filter(asset_types=["ortho_visual"]),  # 8-band SR "ortho_analytic_8b_sr"
    #data_filter.asset_filter(asset_types=["ortho_analytic_sr"]),     # 4-band SR
    ])

    udm2_or = data_filter.or_filter([
        data_filter.asset_filter(asset_types=["ortho_analytic_udm2"]),
        data_filter.asset_filter(asset_types=["ortho_udm2"]),            # some items expose this
    ])
    sfilter = data_filter.and_filter([
        data_filter.std_quality_filter(),
        sr_or,
        udm2_or,
        data_filter.date_range_filter("acquired", gte=datetime.fromisoformat(DATE_START), lte=datetime.fromisoformat(DATE_END)),
        data_filter.range_filter("cloud_cover", lte=MAX_CLOUD),
    ])

    for item in pl.data.search(["PSScene"], geometry=aoi_geom, search_filter=sfilter, limit=10000):
        props = item.get("properties", {})
        items.append({
            "id": item.get("id"),
            "acquired": props.get("acquired"),
            "cloud_cover": props.get("cloud_cover"),
            "satellite_id": props.get("satellite_id"),
            "strip_id": props.get("strip_id"),
            "publishing_stage": props.get("publishing_stage"),
        })

    df = pd.DataFrame(items).sort_values(["cloud_cover", "acquired"])
    print(f"[{aoi_name}] Found {len(df)} items.")

    # Save CSV per AOI
    csv_path = aoi_out / f"search_results_{YEAR}_{aoi_name}.csv"
    df.to_csv(csv_path, index=False)
    print(f"[{aoi_name}] Wrote: {csv_path}")

    # ---- Chunk into batches ----
    def chunk(lst, n):
        for i in range(0, len(lst), n):
            yield lst[i:i+n]
    item_ids = df["id"].tolist()
    batches = list(chunk(item_ids, MAX_ITEMS_PER_ORDER))
    print(f"[{aoi_name}] Total items: {len(item_ids)} across {len(batches)} order(s).")
    return batches


In [8]:

# === Quota charge estimate (per-scene & per-order) ===
# Computes how much area (km²) Planet will deduct from your quota for this order,
# using Planet's Preferred vs Premium charging rules.
# References:
# - Preferred vs Premium minimums: 100 km² per intersecting scene vs actual area. See Orders API FAQ. 
# - New Quota Reservations API can also estimate quota for supported products.
# Docs: https://support.planet.com/hc/en-us/articles/360011216618-Orders-API-FAQ
#       https://docs.planet.com/develop/apis/quota/
from shapely.geometry import shape
from shapely.ops import transform
import pyproj
import pandas as pd
from tqdm import tqdm

PLAN_MODEL = globals().get("PLAN_MODEL", "preferred").lower()  # 'preferred' (E&R Basic) or 'premium'
EQUAL_AREA_CRS = "EPSG:6933"  # World Cylindrical Equal Area



def fetch_item_geometry(item_id: str):
    """Fetch PSScene item geometry via SDK; fallback to Data API request if needed."""
    try:
        item = pl.data.get_item("PSScene", item_id)  # SDK call (sync helper in this notebook's Planet wrapper)
        geom = item.get("geometry") or item.get("_links", {}).get("_self_geometry")  # robust-ish
        if geom:
            return shape(geom)
    except Exception as e:
        pass
    # Fallback to raw HTTP if SDK isn't available
    import os, requests, base64
    api_key = os.getenv("PL_API_KEY") or os.getenv("PL_APIKEY") or os.getenv("PL_API_KEY_ID") or os.getenv("PL_APIKEY_ID")
    if not api_key:
        # try reading the planet auth file written by `planet auth login`
        try:
            import json, pathlib
            auth_path = pathlib.Path.home()/".planet.json"
            if auth_path.exists():
                api_key = json.loads(auth_path.read_text()).get("key")
        except Exception:
            api_key = None
    if not api_key:
        raise RuntimeError("Planet API key not found. Set PL_API_KEY env var or run `!planet auth login`.")
    url = f"https://api.planet.com/data/v1/item-types/PSScene/items/{item_id}"
    resp = requests.get(url, auth=(api_key, ""))
    resp.raise_for_status()
    return shape(resp.json()["geometry"])

def estimate_for_ids(ids: list[str]) -> pd.DataFrame:
    rows = []
    for sid in tqdm(ids, desc="Fetching footprints & intersecting AOI"):
        try:
            scene_geom = fetch_item_geometry(sid)
        except Exception as e:
            # If footprint not available, assume conservative minimum charge if Preferred; else 0
            rows.append({
                "scene_id": sid,
                "intersect_km2": float('nan'),
                "charged_km2": 100.0 if PLAN_MODEL.startswith("pref") else float('nan'),
                "note": "geometry fetch failed; using minimum for Preferred" if PLAN_MODEL.startswith("pref") else "geometry fetch failed"
            })
            continue
        scene_ea = transform(to_ea, scene_geom)
        inter = aoi_ea.intersection(scene_ea)
        inter_area_km2 = inter.area / 1e6  # m² to km²

        if PLAN_MODEL.startswith("pref"):
            charged = max(100.0, inter_area_km2) if inter_area_km2 > 0 else 0.0
        else:
            charged = inter_area_km2

        rows.append({
            "scene_id": sid,
            "intersect_km2": round(inter_area_km2, 3),
            "charged_km2": round(charged, 3),
            "note": ""
        })
    df_est = pd.DataFrame(rows)
    if not df_est.empty:
        total = df_est["charged_km2"].fillna(0).sum()
        print(f"Estimated quota charge for {len(ids)} scene(s) under plan='{PLAN_MODEL}': {total:.2f} km²")
    return df_est

# Prepare transformers
to_ea = pyproj.Transformer.from_crs("EPSG:4326", EQUAL_AREA_CRS, always_xy=True).transform
aoi_files = sorted(AOI_DIR.glob(GEOJSON_PATTERN))
print(f"Batch AOIs in {AOI_DIR} → {len(aoi_files)} files")
if not aoi_files:
    raise FileNotFoundError(f"No GeoJSON files matched: {AOI_DIR / GEOJSON_PATTERN}")

all_results = {}
for i, aoi in enumerate(aoi_files, start=1):
    print(f"\n=== [{i}/{len(aoi_files)}] {aoi.name} ===")
    aoi_geom = read_aoi_geojson(aoi)
    aoi_poly = shape(aoi_geom)
    aoi_ea = transform(to_ea, aoi_poly)

    # Compute per-batch and total estimates BEFORE creating orders
    all_rows = []
    batches = process_one_aoi(aoi)
    for idx, batch_ids in enumerate(batches, start=1):
        print(f"\n=== Estimating order {idx}/{len(batches)} ({len(batch_ids)} items) ===")
        df_batch = estimate_for_ids(batch_ids)
        df_batch["order_idx"] = idx
        order_total = df_batch["charged_km2"].fillna(0).sum()
        print(f"Order {idx} subtotal (km²): {order_total:.2f}")
        all_rows.append(df_batch)

    quota_estimates = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
    quota_estimates["aoi_path"] = aoi
    display_cols = ["order_idx", "scene_id", "intersect_km2", "charged_km2", "note", "aoi_path"]
    quota_estimates = quota_estimates[display_cols].sort_values(["order_idx", "charged_km2"], ascending=[True, False])
    aoi_name = aoi.stem
    aoi_out = OUT_DIR / aoi_name
    csv_path = aoi_out / f"quota_{YEAR}_{aoi_name}.csv"
    quota_estimates.to_csv(csv_path, index=False)
    print(quota_estimates.head(20))


Batch AOIs in D:\Thesis\glacial_lake_detection_thesis\Training\1 Getting geojson for imageries\aoi_geojson_convexhull → 11 files

=== [1/11] cluster_00000_rank_05_lakes_468_area_46.5km2_1.geojson ===
[cluster_00000_rank_05_lakes_468_area_46.5km2_1] Found 69 items.
[cluster_00000_rank_05_lakes_468_area_46.5km2_1] Wrote: planet_downloads_PSScene_rgb_2020\cluster_00000_rank_05_lakes_468_area_46.5km2_1\search_results_2020_cluster_00000_rank_05_lakes_468_area_46.5km2_1.csv
[cluster_00000_rank_05_lakes_468_area_46.5km2_1] Total items: 69 across 1 order(s).

=== Estimating order 1/1 (69 items) ===


Fetching footprints & intersecting AOI: 100%|██████████| 69/69 [00:24<00:00,  2.79it/s]


Estimated quota charge for 69 scene(s) under plan='preferred': 6992.15 km²
Order 1 subtotal (km²): 6992.15
    order_idx                 scene_id  intersect_km2  charged_km2 note  \
60          1  20200924_053150_64_1067        132.655      132.655        
57          1  20200918_053458_06_106d        129.343      129.343        
46          1  20200908_050721_26_2259        124.526      124.526        
4           1     20200911_031609_1054        105.625      105.625        
0           1     20200909_053014_1004          3.480      100.000        
1           1     20200910_031619_1052         12.468      100.000        
2           1     20200910_031620_1052          4.638      100.000        
3           1     20200911_031608_1054         27.265      100.000        
5           1   20200913_031543_1_0f21          0.923      100.000        
6           1     20200913_031543_0f21         10.397      100.000        
7           1     20200913_031544_0f21          0.302      100.000  

Fetching footprints & intersecting AOI: 100%|██████████| 101/101 [00:36<00:00,  2.79it/s]


Estimated quota charge for 101 scene(s) under plan='preferred': 10801.41 km²
Order 1 subtotal (km²): 10801.41
    order_idx                 scene_id  intersect_km2  charged_km2 note  \
74          1  20200922_055414_50_227c        252.841      252.841        
44          1  20200923_050457_66_222c        236.436      236.436        
43          1  20200923_050455_45_222c        172.453      172.453        
75          1     20200925_052920_1006        158.763      158.763        
93          1     20200907_053050_1038        143.964      143.964        
57          1  20200922_050013_56_2212        143.834      143.834        
40          1     20200923_031317_0f36        139.283      139.283        
70          1     20200925_052921_1006        130.983      130.983        
33          1     20200922_031155_104a        128.505      128.505        
80          1     20200928_031650_0f2e        122.968      122.968        
41          1     20200923_031318_0f36        116.994      116.99

...

Estimated quota charge for 109 scene(s) under plan='preferred': 11191.76 km²
Order 1 subtotal (km²): 11191.76
     order_idx                 scene_id  intersect_km2  charged_km2 note  \
14           1  20200919_045654_59_220b        167.306      167.306        
95           1  20200917_045649_33_2278        167.306      167.306        
82           1  20200908_045709_78_2231        161.064      161.064        
93           1  20200910_055211_67_1061        128.521      128.521        
42           1     20200924_052424_100a        119.128      119.128        
107          1     20200912_052138_1025        115.532      115.532        
74           1     20200913_052457_1040        114.904      114.904        
83           1  20200913_052425_15_106c        110.453      110.453        
12           1  20200919_045302_38_2206        107.548      107.548        
0            1     20200901_052250_1011         90.984      100.000        
1            1     20200902_031318_0f24          4.890

Fetching footprints & intersecting AOI: 100%|██████████| 105/105 [00:35<00:00,  2.94it/s]


Estimated quota charge for 105 scene(s) under plan='preferred': 11064.15 km²
Order 1 subtotal (km²): 11064.15
     order_idx                 scene_id  intersect_km2  charged_km2 note  \
62           1  20200918_045757_86_2271        193.798      193.798        
10           1  20200919_045257_95_2206        182.832      182.832        
94           1  20200917_045646_94_2278        166.728      166.728        
103          1     20200925_052538_0f28        147.358      147.358        
102          1  20200925_052501_90_106c        146.541      146.541        
42           1     20200924_052346_0f4e        143.586      143.586        
80           1     20200929_053654_1105        136.544      136.544        
13           1  20200919_045652_38_220b        131.211      131.211        
99           1  20200929_055537_45_1058        129.337      129.337        
82           1  20200908_045707_43_2231        125.067      125.067        
55           1     20200926_030618_0f44        121.934

Fetching footprints & intersecting AOI: 100%|██████████| 102/102 [00:34<00:00,  2.96it/s]

Estimated quota charge for 102 scene(s) under plan='preferred': 10281.67 km²
Order 1 subtotal (km²): 10281.67
    order_idx                 scene_id  intersect_km2  charged_km2 note  \
1           1  20200901_045513_72_2277        136.402      136.402        
24          1  20200921_055301_65_105a        129.323      129.323        
0           1  20200901_045114_37_2206        115.949      115.949        
2           1     20200913_030532_0f49         45.926      100.000        
3           1     20200913_030533_0f49         56.042      100.000        
4           1     20200917_030502_1049         12.538      100.000        
5           1     20200917_030505_1049         53.392      100.000        
6           1     20200919_030427_0f46         24.328      100.000        
7           1     20200919_030428_0f46         43.050      100.000        
8           1     20200919_030429_0f46         44.783      100.000        
9           1     20200919_030430_0f46         22.843      100.00

In [9]:
path_list = []
for aoi_file in aoi_files:
    aoi_name = aoi_file.stem
    aoi_out = OUT_DIR / aoi_name
    csv_path = aoi_out / f"quota_{YEAR}_{aoi_name}.csv"
    path_list.append(csv_path)
frames = [pd.read_csv(p, low_memory=False).assign(source_file=p.name) for p in path_list]

# concatenate into a single DataFrame
df = pd.concat(frames, ignore_index=True, sort=False)
df.to_csv(OUT_DIR / "quota_concatenated.csv", index=False)

In [7]:
df = pd.read_csv(OUT_DIR / "quota_concatenated.csv")

In [8]:
order_scenes = ["20200922_050042_44_2278",
                "20200908_050721_26_2259",
                "20200918_045757_86_2271",
                "20200922_055346_27_227c",
                "20200919_045654_59_220b",
                "20200901_045513_72_2277",
                "20200925_050740_48_222f",
                "20200922_055934_31_241c",
                "20200912_055505_07_241c",
                "20200919_045639_12_220b",
                "20200922_055414_50_227c"]

order_df = df[df["scene_id"].isin(order_scenes)]
order_df = order_df[order_df["intersect_km2"] > 100]


In [9]:
order_df

,order_idx,scene_id,intersect_km2,charged_km2,note,aoi_path,source_file
2,1,20200908_050721_26_2259,124.526,124.526,NaN,D:\Thesis\glacial_lake_detection_thesis\Traini...,quota_2020_cluster_00000_rank_05_lakes_468_are...
69,1,20200922_055414_50_227c,252.841,252.841,NaN,D:\Thesis\glacial_lake_detection_thesis\Traini...,quota_2020_cluster_00000_rank_05_lakes_468_are...
170,1,20200925_050740_48_222f,400.697,400.697,NaN,D:\Thesis\glacial_lake_detection_thesis\Traini...,quota_2020_cluster_00040_rank_01_lakes_1031_ar...
295,1,20200922_055934_31_241c,352.854,352.854,NaN,D:\Thesis\glacial_lake_detection_thesis\Traini...,quota_2020_cluster_00040_rank_01_lakes_1031_ar...
393,1,20200912_055505_07_241c,249.563,249.563,NaN,D:\Thesis\glacial_lake_detection_thesis\Traini...,quota_2020_cluster_00040_rank_01_lakes_1031_ar...
520,1,20200922_050042_44_2278,118.154,118.154,NaN,D:\Thesis\glacial_lake_detection_thesis\Traini...,quota_2020_cluster_00085_rank_02_lakes_1026_ar...
570,1,20200919_045639_12_220b,293.568,293.568,NaN,D:\Thesis\glacial_lake_detection_thesis\Traini...,quota_2020_cluster_00089_rank_04_lakes_539_are...
659,1,20200922_055346_27_227c,130.725,130.725,NaN,D:\Thesis\glacial_lake_detection_thesis\Traini...,quota_2020_cluster_00122_rank_13_lakes_186_are...
704,1,20200919_045654_59_220b,167.306,167.306,NaN,D:\Thesis\glacial_lake_detection_thesis\Traini...,quota_2020_cluster_00175_rank_20_lakes_108_are...
813,1,20200918_045757_86_2271,193.798,193.798,NaN,D:\Thesis\glacial_lake_detection_thesis\Traini...,quota_2020_cluster_00176_rank_06_lakes_364_are...


In [28]:
order_scenes = ["20200923_050442_18_222c"]

order_df = df[df["scene_id"].isin(order_scenes)]
order_df = order_df[order_df["intersect_km2"] > 100]
order_df

,order_idx,scene_id,intersect_km2,charged_km2,note,aoi_path,source_file
397,1,20200923_050442_18_222c,197.062,197.062,NaN,C:\Users\gg\Documents\MS Data Science\Thesis\g...,quota_2020_cluster_00040_rank_01_lakes_1031_ar...


In [10]:
for idx, row in order_df.iterrows():
    print(row["scene_id"])


20200908_050721_26_2259
20200922_055414_50_227c
20200925_050740_48_222f
20200922_055934_31_241c
20200912_055505_07_241c
20200922_050042_44_2278
20200919_045639_12_220b
20200922_055346_27_227c
20200919_045654_59_220b
20200918_045757_86_2271
20200901_045513_72_2277


In [11]:
for idx, row in order_df.iterrows():
    aoi_geom = read_aoi_geojson(Path(row["aoi_path"]))

    tools = [
        order_request.clip_tool(aoi_geom),
        order_request.reproject_tool(REPROJECT_EPSG),
        order_request.file_format_tool("COG"),
    ]

    delivery_cfg = None
    if ZIP_EACH_ORDER:
        delivery_cfg = order_request.delivery(archive_type="zip", single_archive=True)

    order_ids = []
    
    order_name = f"PSScene_{YEAR}_AOI_{idx:02d}"
    print(f"Creating order with scene ID {row["scene_id"]}")

    req = order_request.build_request(
        name=order_name,
        products=[order_request.product(item_ids=[row["scene_id"]], product_bundle=PRIMARY_BUNDLE, item_type="PSScene",
                                            fallback_bundle=FALLBACK_BUNDLES or None)],
        tools=tools,
        delivery=delivery_cfg
    )
    order = pl.orders.create_order(req)
    oid = order["id"]
    order_ids.append(oid)
    print("  created:", oid)

    with reporting.StateBar(state="creating") as bar:
        pl.orders.wait(oid, callback=bar.update_state, max_attempts=WAIT_MAX_ATTEMPTS)

    order_dir = OUT_DIR / f"order_{idx:02d}_{oid}"
    order_dir.mkdir(parents=True, exist_ok=True)
    print("  downloading to:", order_dir)
    pl.orders.download_order(oid, directory=str(order_dir), overwrite=True)

    order_ids


Creating order with scene ID 20200908_050721_26_2259
  created: 203ef10e-6bf1-435d-a34e-7f4f61d71c3f


13:11 - order  - state: success 


  downloading to: planet_downloads_PSScene_rgb_2020\order_02_203ef10e-6bf1-435d-a34e-7f4f61d71c3f
Creating order with scene ID 20200922_055414_50_227c
  created: 811cb477-f355-4d7b-8d8f-0e002da49e0e


09:56 - order  - state: success 


  downloading to: planet_downloads_PSScene_rgb_2020\order_69_811cb477-f355-4d7b-8d8f-0e002da49e0e
Creating order with scene ID 20200925_050740_48_222f
  created: 61ea2878-b830-4e9c-9a6e-60ec1966c695


...

  downloading to: planet_downloads_PSScene_rgb_2020\order_704_3b70de75-8dce-4e63-a7a6-a4d7c7e24bf0
Creating order with scene ID 20200918_045757_86_2271
  created: cd8d4460-a5ce-47f7-8e80-8446b8e25f9b


10:46 - order  - state: success 


  downloading to: planet_downloads_PSScene_rgb_2020\order_813_cd8d4460-a5ce-47f7-8e80-8446b8e25f9b
Creating order with scene ID 20200901_045513_72_2277
  created: 704dba7f-9bc0-499f-9f39-6b6cc693d9fc


15:07 - order  - state: success 


  downloading to: planet_downloads_PSScene_rgb_2020\order_918_704dba7f-9bc0-499f-9f39-6b6cc693d9fc
